In [2]:
import cellxgene_census
import pandas as pd
import scanpy as sc
import numpy as np
import json
import anndata as ad
import re

In [1]:
cell_gene_dir = "../../data/censusxgene"
# cell_types = [
#     "malignant cell",
#     "luminal epithelial cell of mammary gland",
#     "basal-myoepithelial cell of mammary gland",
#     "fibroblast of mammary gland",
#     "macrophage",
#     "T cell",
#     "B cell"
# ]

tissues = ['breast', 'lung', 'kidney', 'bladder organ'] # 15M cells
tissues_general = ['breast', 'lung', 'kidney', 'bladder organ'] # 17.9M cells
# simplified & only cell types that are in all tissues - # 12M cells
# only cell types that are in all tissueas - # 5M cells
 
with open("../../data/protein_coding_genes.txt", "r") as f:
    protein_coding = [line.strip() for line in f]

In [53]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    obs_df = cellxgene_census.get_obs(
        census,
        "homo_sapiens",
        value_filter=f"tissue_general in {tissues_general} and suspension_type == 'cell'" # and cell_type in {cell_types}"
    )

In [54]:
obs_df

,soma_joinid,dataset_id,assay,assay_ontology_term_id,cell_type,cell_type_ontology_term_id,development_stage,development_stage_ontology_term_id,disease,disease_ontology_term_id,...,tissue,tissue_ontology_term_id,tissue_type,tissue_general,tissue_general_ontology_term_id,raw_sum,nnz,raw_mean_nnz,raw_variance_nnz,n_measured_vars
0,31188,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,18th week post-fertilization stage,HsapDv:0000055,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,5780.0,2236,2.584973,42.338638,16035
1,31189,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,15th week post-fertilization stage,HsapDv:0000052,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,2087.0,1357,1.537951,5.754643,16035
2,31190,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,Schwann cell,CL:0002573,15th week post-fertilization stage,HsapDv:0000052,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,14733.0,4268,3.451968,107.457969,16035
3,31191,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,immature Schwann cell,CL:0002377,Carnegie stage 19,HsapDv:0000026,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,5174.0,1797,2.879243,53.412470,16035
4,31192,703f00e6-b996-48e5-bc34-00c41b9876f4,10x 5' v1,EFO:0011025,neuron,CL:0000540,Carnegie stage 19,HsapDv:0000026,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,8107.0,3033,2.672931,31.431249,16035
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17890872,156173185,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,"CD8-positive, alpha-beta T cell",CL:0000625,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,4561.0,2016,2.262401,53.842278,60606
17890873,156173186,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,capillary endothelial cell,CL:0002144,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,8823.0,3366,2.621212,68.019033,60606
17890874,156173187,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,endothelial cell of artery,CL:1000413,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,12442.0,3943,3.155465,133.749290,60606
17890875,156173188,53d208b0-2cfd-4366-9866-c3c6114081bc,10x 3' v3,EFO:0009922,macrophage,CL:0000235,60-year-old stage,HsapDv:0000154,normal,PATO:0000461,...,lung,UBERON:0002048,tissue,lung,UBERON:0002048,38498.0,6121,6.289495,994.178270,60606


In [55]:
cell_types = set(obs_df['cell_type'].values)
len(cell_types)

360

In [56]:
counts = (
    obs_df
    .groupby(["tissue_general", "tissue", "cell_type"], observed=True)
    .size()
    .reset_index(name="count")
)

counts.to_csv(f"{cell_gene_dir}/cell_counts_per_group_path.csv", index=False)

In [57]:
np.random.seed(42)

cells_per_type = 120000
selected_ids = []

for tissue in tissues_general:
    subset_ct = obs_df[obs_df["tissue_general"] == tissue]
    n_available = len(subset_ct)
    n_sample = min(cells_per_type, n_available)
    
    ids = np.random.choice(subset_ct["soma_joinid"].values, n_sample, replace=False)
    selected_ids.extend(ids)

    print(tissue, n_available, n_sample)

breast 6962997 120000
lung 9156186 120000
kidney 1563491 120000
bladder organ 208203 120000


In [58]:
obs_df_filtered = obs_df[obs_df["soma_joinid"].isin(selected_ids)]

In [59]:
min_cells = 200
counts = obs_df_filtered["cell_type"].value_counts()
valid_types = counts[counts >= min_cells].index
valid_types = list(valid_types)
if 'unknown' in valid_types:
    valid_types.remove('unknown')
obs_df_filtered = obs_df_filtered[obs_df_filtered["cell_type"].isin(valid_types)]

In [60]:
np.random.seed(42)

cells_per_type = 100000
selected_ids_filtered = []

for tissue in tissues_general:
    subset_ct = obs_df_filtered[obs_df_filtered["tissue_general"] == tissue]
    n_available = len(subset_ct)
    n_sample = min(cells_per_type, n_available)
    
    ids = np.random.choice(subset_ct["soma_joinid"].values, n_sample, replace=False)
    selected_ids_filtered.extend(ids)

    print(tissue, n_available, n_sample)

breast 117762 100000
lung 100016 100000
kidney 106679 100000
bladder organ 119997 100000


In [61]:
cell_types = set(obs_df_filtered['cell_type'].values)

len(cell_types)

143

In [62]:
cell_types

{'B cell',
 'CD14-positive monocyte',
 'CD14-positive, CD16-positive monocyte',
 'CD16-negative, CD56-bright natural killer cell, human',
 'CD16-positive, CD56-dim natural killer cell, human',
 'CD1c-positive myeloid dendritic cell',
 'CD4-positive helper T cell',
 'CD4-positive, CD25-positive, CCR4-positive, alpha-beta regulatory T cell',
 'CD4-positive, CD25-positive, alpha-beta regulatory T cell',
 'CD4-positive, alpha-beta T cell',
 'CD8-positive, alpha-beta T cell',
 'CD8-positive, alpha-beta cytotoxic T cell',
 'CD8-positive, alpha-beta memory T cell',
 'CD8-positive, alpha-beta regulatory T cell',
 'IgA plasma cell',
 'IgG plasma cell',
 'T cell',
 'T follicular helper cell',
 'Tc1 cell',
 'abnormal cell',
 'activated CD4-positive, alpha-beta T cell, human',
 'activated type II NK T cell',
 'alternatively activated macrophage',
 'alveolar adventitial fibroblast',
 'alveolar capillary type 1 endothelial cell',
 'alveolar macrophage',
 'alveolar type 1 fibroblast cell',
 'basal ce

In [63]:
with cellxgene_census.open_soma(census_version="2025-11-08") as census:
    cell_adata = cellxgene_census.get_anndata(
        census,
        "homo_sapiens",
        obs_coords=selected_ids_filtered,
        column_names=["assay", "cell_type", "donor_id", "tissue", "tissue_general", "suspension_type", "disease"]
    )

/tmp/ipykernel_2064824/3626444250.py:2: FutureWarning: The argument `column_names` is deprecated and will be removed in a future release. Please use `obs_column_names` and `var_column_names` instead.
  cell_adata = cellxgene_census.get_anndata(
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/scratch/2370352/conda/envs/myenv/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [64]:
cols_you_want = [
    "soma_joinid",
    "cell_type",
    "donor_id",
    "tissue",
    "tissue_general",
    "suspension_type",
]

cell_adata.obs = cell_adata.obs[cols_you_want]
cell_adata.obs["cell_type"] = cell_adata.obs["cell_type"].cat.remove_unused_categories()

In [65]:
cell_adata.obs['cell_type'].value_counts()[:25]

cell_type
bladder urothelial cell                                       39099
fibroblast                                                    29415
macrophage                                                    20175
epithelial cell of proximal tubule                            16613
CD8-positive, alpha-beta T cell                               15877
CD4-positive, alpha-beta T cell                               12729
fibroblast of mammary gland                                   12092
luminal adaptive secretory precursor cell of mammary gland    11199
monocyte                                                      11018
alveolar macrophage                                           10915
malignant cell                                                10714
pulmonary alveolar type 2 cell                                 8504
kidney loop of Henle thick ascending limb epithelial cell      8323
T cell                                                         8257
luminal hormone-sensing cell of mammar

In [66]:
sc.pp.normalize_total(cell_adata, target_sum=1e4) #, inplace=False)
sc.pp.log1p(cell_adata)

In [67]:
print(cell_adata)
print("obs columns:", cell_adata.obs.columns)
print("var shape:", cell_adata.var.shape)
print("X type:", type(cell_adata.X))

AnnData object with n_obs × n_vars = 400000 × 61497
    obs: 'soma_joinid', 'cell_type', 'donor_id', 'tissue', 'tissue_general', 'suspension_type'
    var: 'soma_joinid', 'feature_id', 'feature_name', 'feature_type', 'feature_length', 'nnz', 'n_measured_obs'
    uns: 'log1p'
obs columns: Index(['soma_joinid', 'cell_type', 'donor_id', 'tissue', 'tissue_general',
       'suspension_type'],
      dtype='object')
var shape: (61497, 7)
X type: <class 'scipy.sparse._csr.csr_matrix'>


In [68]:
vocab_path = '/scratch/2370352/my-research/papers/scgpt/save/whole_human/vocab.json'

with open(vocab_path, "r") as f:
    vocab = json.load(f)

model_genes = list(vocab.keys())

print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(model_genes[:10])  # przykładowe pierwsze 10 genów

Liczba genów w scGPT vocab: 60697
['RP5-973N23.5', 'RP11-182N22.10', 'CTB-53D8.3', 'RP11-348N17.2', 'RP11-205M20.8', 'RP11-326C3.17', 'RP11-439H13.3', 'RP11-413H22.3', 'GET1-SH3BGR', 'CH17-476P10.1']


In [69]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 61497
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 38598
Liczba genów w adata nie w vocab: 21430
Liczba genów w vocab nie w adata: 22099
Przykłady genów wspólnych: ['SHISAL2A', 'LINC02673', 'SNRPN', 'UNC5B-AS1', 'DNLZ', 'DCTN6', 'KLHL7', 'CLIC4P2', 'GRB14', 'RNU6-29P']
Przykłady genów w adata ale nie w vocab: ['ENSG00000261278', 'ENSG00000281974', 'ENSG00000279114', 'ENSG00000280649', 'ENSG00000231855', 'ENSG00000277795', 'ENSG00000289296', 'ENSG00000257746', 'ENSG00000290870', 'ENSG00000274591']
Przykłady genów w vocab ale nie w adata: ['RP3-495K2.1', 'RP5-1180D12.1', 'RP11-749H20.2', 'RP11-327I22.3', 'RP11-366N18.2', 'RP11-394B2.6', 'Y_RNA_ENSG00000201668', 'SIGLEC5_ENSG00000268500', 'RP11-452G18.2', 'RP11-274B21.1']


In [70]:
gene_info = pd.read_csv("/scratch/2370352/my-research/data/gene_info_table.csv")  # lub pełna ścieżka

# Stwórz słownik: ensembl_id -> gene_name
ensg_to_symbol = dict(zip(gene_info['ensembl_id'], gene_info['gene_name']))

print(list(ensg_to_symbol.items())[:10])

[('ENSG00000000003', 'TSPAN6'), ('ENSG00000000005', 'TNMD'), ('ENSG00000000419', 'DPM1'), ('ENSG00000000457', 'SCYL3'), ('ENSG00000000460', 'C1orf112'), ('ENSG00000000938', 'FGR'), ('ENSG00000000971', 'CFH'), ('ENSG00000001036', 'FUCA2'), ('ENSG00000001084', 'GCLC'), ('ENSG00000001167', 'NFYA')]


In [71]:
# jeśli w adata masz geny zapisane jako ENSG, mapujemy je
mapped_genes = []

for g in cell_adata.var['feature_name']:
    if g in ensg_to_symbol:
        mapped_genes.append(ensg_to_symbol[g])
    else:
        mapped_genes.append(g)  # zachowaj tak jak jest, np. już symboliczny gen

# podmieniamy w adata.var
cell_adata.var['feature_name_mapped'] = mapped_genes

In [72]:
# adata_genes = lista genów z adata
# model_genes = lista genów z scGPT vocab

adata_genes = cell_adata.var['feature_name_mapped'].tolist()

# zamień na sety dla szybkiego porównania
adata_set = set(adata_genes)
model_set = set(model_genes)

# wspólne geny
common_genes = adata_set & model_set

# geny w adata, których nie ma w scGPT
missing_in_model = adata_set - model_set

# geny w scGPT, których nie ma w adata
missing_in_adata = model_set - adata_set

print(f"Liczba genów w adata: {len(adata_genes)}")
print(f"Liczba genów w scGPT vocab: {len(model_genes)}")
print(f"Liczba genów wspólnych: {len(common_genes)}")
print(f"Liczba genów w adata nie w vocab: {len(missing_in_model)}")
print(f"Liczba genów w vocab nie w adata: {len(missing_in_adata)}")

# przykładowe geny
print("Przykłady genów wspólnych:", list(common_genes)[:10])
print("Przykłady genów w adata ale nie w vocab:", list(missing_in_model)[:10])
print("Przykłady genów w vocab ale nie w adata:", list(missing_in_adata)[:10])

Liczba genów w adata: 61497
Liczba genów w scGPT vocab: 60697
Liczba genów wspólnych: 47856
Liczba genów w adata nie w vocab: 11611
Liczba genów w vocab nie w adata: 12841
Przykłady genów wspólnych: ['SHISAL2A', 'LINC02673', 'SNRPN', 'UNC5B-AS1', 'DNLZ', 'RP3-495K2.1', 'DCTN6', 'KLHL7', 'CLIC4P2', 'RP11-366N18.2']
Przykłady genów w adata ale nie w vocab: [nan, 'LOHAN2', 'AL133351.5', 'AL512329.2', 'CD300LD-AS1', 'ENSG00000289296', 'ENSG00000285599', 'ENSG00000290870', 'UQCC4', 'RPL7AP82']
Przykłady genów w vocab ale nie w adata: ['RP11-137H2.4', 'RP11-696F10.1', 'RP3-475N16.1', 'RP11-109E12.4', 'RP11-114F10.2', 'RP11-798M19.3', 'RP11-519G16.3', 'RP11-1030E3.2', 'RP5-1180D12.1', 'Y_RNA_ENSG00000251811']


In [73]:
np.random.seed(42)

all_indices = np.arange(cell_adata.n_obs)
np.random.shuffle(all_indices)

split = int(0.9 * len(all_indices))

train_indices = all_indices[:split]
test_indices  = all_indices[split:]

train_adata = cell_adata[train_indices].copy()
test_adata  = cell_adata[test_indices].copy()

In [74]:
train_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_path_train.h5ad")
test_adata.write("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_path_test.h5ad")


In [75]:
adata = ad.read_h5ad("/scratch/2370352/my-research/adapter_premium/data_new/blkb_common_path_test.h5ad")


In [76]:
adata.obs['cell_type'].value_counts()

cell_type
bladder urothelial cell                          3880
fibroblast                                       2875
macrophage                                       2116
epithelial cell of proximal tubule               1672
CD8-positive, alpha-beta T cell                  1543
                                                 ... 
naive B cell                                       18
mesonephric nephron tubule epithelial cell         16
erythrocyte                                        16
fibroblast of breast                               15
progenitor cell of mammary luminal epithelium       9
Name: count, Length: 143, dtype: int64